
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Embeddings and Vector Search

## Introduction

The effectiveness of any retrieval-augmented generation (RAG) system hinges on one critical factor: the quality of its retrieval pipeline. Before we can retrieve relevant information, we must first transform unstructured text into numerical representations called **embeddings** and store them in specialized **vector databases**. In this lesson, we'll explore the complete data preparation lifecycle—from understanding how **embedding models** convert text into vectors, to leveraging **vector similarity algorithms** for efficient search. We'll also examine advanced retrieval techniques like hybrid search and reranking that significantly improve result quality. Finally, we'll discover how **Mosaic AI Vector Search** brings these components together within the Databricks Data Intelligence Platform, delivering a secure, serverless vector database solution.

## Lesson Objectives

By the end of this lesson, you will be able to:

* Identify the key characteristics of embedding vectors and evaluate criteria for selecting appropriate embedding models.
* Compare similarity search, full-text search, and hybrid search methodologies to determine optimal retrieval strategies.
* Analyze the trade-offs between exact and approximate search algorithms in production environments.
* Explain how reranking improves context precision and reduces hallucinations in RAG applications.
* Describe the ingestion modes, governance model, and architectural benefits of Mosaic AI Vector Search.

## A. Core Concepts of Embeddings

In this section, we'll establish the foundational knowledge you need to work with embeddings—the mathematical backbone of modern information retrieval. We'll explore **how unstructured text transforms into vector representations**, examine why selecting the right model matters for your specific domain, and understand the critical importance of alignment between query and document spaces.

### A1. Defining Embeddings

An embedding is a numerical representation of content, typically generated by a deep learning model. These models convert high-dimensional unstructured data (like text) into lower-dimensional vectors—arrays of floating-point numbers that capture semantic meaning. The key characteristic that makes embeddings powerful is their ability to map similar concepts close together in vector space. Words or phrases with related meanings cluster near each other, enabling systems to identify conceptual relationships even when exact keywords don't match.

### A2. Multimodal Context

While **this lesson focuses on unstructured text**, it's worth noting that embeddings extend far beyond words. Multimodal models like GPT-4o and Gemini 1.5 can process and embed images, audio, and text into a unified vector space. This capability unlocks cross-modal retrieval scenarios—imagine using a text query to find semantically relevant images, or searching audio content with written descriptions.

### A3. Embedding Models

An embedding model is a specialized machine learning model (typically a deep neural network) designed to convert high-dimensional unstructured data—such as text, images, or audio—into lower-dimensional numerical vectors. **Think of it as a translator that converts human-readable content into machine-readable lists of floating-point numbers, ensuring that inputs with similar meanings produce vectors that are mathematically close to each other.**

Selecting the right embedding model is a critical architectural decision that impacts retrieval quality. Consider these key factors:

- **Vocabulary Size and Domain:** Some models train on general web text, while others specialize in specific domains like finance, medicine, or legal documents. Domain-specific models often deliver superior results for specialized content.
- **Context Window:** Every model has a maximum input token limit. Text exceeding this limit gets truncated or ignored, making effective chunking strategies essential for long documents.
- **Dimensions:** Higher-dimensional vectors (larger arrays) capture more nuance and semantic detail but increase storage costs and retrieval latency. Balance precision needs with operational constraints.

![03-vectorization](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-en_us-1.0.1/images/03-vectorization.png)

*Figure 1. This illustration demonstrates how data chunks are processed by an embedding model to generate vectors. If the input data exceeds the model's context window limit, the excess content is omitted, which can impact the completeness of the resulting embeddings.*

### A4. Embedding Alignment

For retrieval to work effectively, your embedding model must represent both source documents and user queries in the same vector space. If the model trained primarily on long-form documents but your application uses short, informal queries, the vector representations may not align well—leading to poor retrieval results. The best practice is straightforward: **use the same embedding model for both indexing documents and processing queries**. This ensures they exist in the same mathematical space and can be meaningfully compared.

## B. Vector Stores and Search Mechanics

Once we've converted unstructured data into embeddings, we need specialized storage that can handle high-dimensional vectors and perform efficient similarity queries. In this section, we'll examine the distinctive architecture of vector databases and how they differ from traditional relational systems. We'll also explore the search algorithms and metrics used to retrieve semantically relevant information at scale.

### B1. The Role of Vector Databases

A vector database is purpose-built to store and retrieve high-dimensional vectors efficiently. Unlike traditional databases designed for exact matches (think SQL WHERE clauses), vector databases excel at similarity searches—finding items that are conceptually related rather than identical. They maintain standard database capabilities like Create-Read-Update-Delete (CRUD) operations while introducing specialized indexing structures optimized for vector operations.

### B2. Search Methods

Different search methods serve different retrieval needs:

- **Similarity Search:** This method retrieves content based on semantic correlation rather than exact word matching. It enables natural language queries like "how to deal with anxiety" to surface relevant results that may use different terminology, such as "coping with PTSD" or "managing stress."
- **Full-Text Search:** This traditional approach relies on keyword matching. It excels at finding specific terms like part numbers, product codes, or proper nouns, but fails to capture semantic intent or recognize synonyms.
- **Hybrid Search:** This powerful approach combines vector similarity search with keyword-based search. By leveraging both semantic understanding and exact keyword matching, hybrid search typically delivers higher retrieval accuracy than either method alone.

### B3. Distance and Similarity Metrics

To determine how "similar" two vectors are, we use two main types of metrics—**distance metrics** and **similarity metrics**—each suited to different retrieval scenarios. **Distance metrics** are used when you want to quantify how far apart two vectors are in space—ideal for clustering, outlier detection, or when magnitude matters. **Similarity metrics** are used when you want to know how closely aligned two vectors are in direction—ideal for semantic search, document retrieval, and most NLP applications where meaning is more important than scale.

**Distance Metrics**  
- **Euclidean Distance (L2):** Measures the straight-line distance between two points in vector space. A *lower* value means vectors are more similar. Use this when you care about the absolute difference in all dimensions, such as clustering or anomaly detection.
- **Manhattan Distance (L1):** Sums the absolute differences across all dimensions. A *lower* value means vectors are closer. This is useful when differences along each axis are equally important, such as in grid-based or sparse data.

**Similarity Metrics**  
- **Cosine Similarity:** Measures the cosine of the angle between two vectors. A *higher* score means greater similarity. This is the most popular metric for text embeddings because it focuses on orientation (semantic meaning) rather than magnitude, making it robust to document length or scale differences.

<!-- 
<img src="../Includes/images/03-vector-similarities.png" alt="Distance and similarity metrics" /> -->

![03-vector-similarities](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-en_us-1.0.1/images/03-vector-similarities.png)

*Figure 2. This diagram visualizes the distance metrics and similarity metrics listed above*

### B4. Search Strategies

Two primary strategies balance accuracy and performance:

- **K-Nearest Neighbors (KNN):** An exact search method that calculates the distance between the query vector and *every* vector in the database. While highly accurate, it's computationally expensive and doesn't scale well to large datasets—imagine comparing your query against millions of documents one by one.
- **Approximate Nearest Neighbors (ANN):** A strategy that trades a small amount of accuracy for dramatic speed gains. ANN uses sophisticated indexing algorithms like **HNSW** (Hierarchical Navigable Small Worlds) or **FAISS** (Facebook AI Similarity Search) to navigate vector space efficiently, checking only a subset of vectors while still finding highly relevant results.

## C. Precision, Quality, and Reranking

While vector databases provide a powerful mechanism for finding semantically similar content, they're not without limitations. In this section, we'll address the nuances of embedding quality and the potential gap between mathematical similarity and true semantic relevance. We'll also introduce reranking as a critical post-retrieval step that refines results and improves the accuracy of context provided to language models.

### C1. Embedding Quality and Limitations

Here's a crucial insight: **similarity does not equal semantic relevance**. A document might be mathematically close to your query in vector space yet remain factually irrelevant or contextually inappropriate. Embedding quality depends heavily on the model, its training data, and how well it aligns with your specific domain. Improperly prepared data or a mismatch between the model's training corpus and your application's content can lead to poor retrieval performance and "lost" information.

Another common scenario involves selecting a subset of documents rather than using all results from a similarity search. When you need to limit the number of documents—perhaps due to token constraints or processing costs—you'll want to ensure the most relevant documents rise to the top. This is where reranking becomes essential.

### C2. The Reranking Process

To bridge the precision gap in initial retrieval, we add a **reranker** to the pipeline:

1. **Initial Retrieval:** The vector store retrieves a broad set of candidate documents (typically the top 20 to 50) using fast ANN algorithms.
1. **Reranking:** A specialized model (often a Cross-Encoder) evaluates the actual relevance of each candidate document against the specific query, considering their relationship in detail.
1. **Reordering:** Documents are re-sorted based on the reranker's relevance scores, placing the most pertinent information at the top for the language model to process.

<!-- <img src="../Includes/images/03-reranking.png" alt="Reranking process" width="500" /> -->

![03-reranking](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-en_us-1.0.1/images/03-reranking.png)

*Figure 3. This diagram shows the reranking process works*

### C3. Benefits and Trade-offs

Reranking introduces important considerations:

- **Benefits:** Reranking significantly improves the accuracy of context provided to the language model, which directly reduces hallucinations and improves response quality. By refining the initial retrieval results, you ensure the most relevant information reaches the generation stage.
- **Trade-offs:** Adding a reranker increases both latency and cost in your retrieval pipeline. The reranking model must process the query and candidate documents in real-time, adding computational overhead. Balance these costs against the quality improvements for your specific use case.

## D. Mosaic AI Vector Search - Features and Architecture

Implementing a robust vector database infrastructure can be complex, but Databricks simplifies this process with Mosaic AI Vector Search. In this section, we'll explore the service's architecture and highlight its seamless integration with Delta Lake for automatic data syncing. We'll also examine the unified governance model under Unity Catalog, which ensures secure and managed access to vector indexes.

<!-- <img src="../Includes/images/03-vector-search-components.png" alt="Mosaic AI Vector Search components" /> -->

![03-vector-search-components](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-en_us-1.0.1/images/03-vector-search-components.png)

*Figure 4. This diagram shows the main components of Mosaic AI Vector Search* 

### D1. Product Overview

**Mosaic AI Vector Search** is a **vector database solution integrated directly into the Databricks Lakehouse**. This scalable, low-latency service stores vector representations of your data alongside their metadata, enabling real-time similarity search through a REST API and Python client. It's purpose-built to optimize retrieval for RAG applications, eliminating the need to manage separate vector database infrastructure.

### D2. Delta Sync and Indexing

One of the most powerful features of Mosaic AI Vector Search is its tight integration with **Delta Lake**. Through the **Delta Sync API**, your vector index automatically syncs with a source Delta table. When you add, update, or delete data in the source table, the vector index updates automatically—ensuring your retrieval system always reflects the most current data without manual intervention. This eliminates the operational burden of keeping embeddings synchronized with your source data.

### D3. Management and Ingestion Modes

Mosaic AI Vector Search offers three flexible approaches for ingesting and managing embeddings, allowing you to choose the level of control that fits your needs:

1. **Managed Embeddings (Delta Sync):** You provide a source Delta table containing raw text, and Databricks handles the rest. The system automatically computes embeddings using a configured **Mosaic AI Model Serving** endpoint (such as a Foundation Model API), processes new data, and updates the index—all without requiring you to manage the embedding pipeline.

1. **Self-Managed Embeddings (Delta Sync):** You compute embeddings using your own custom pipelines and store them in a Delta table. The Vector Search index syncs with this table, indexing the pre-computed vectors you provide. This gives you full control over the embedding process while still benefiting from automatic synchronization.

1. **Direct Access CRUD API:** You can interact directly with the Vector Search index using the REST API or Python SDK. This allows you to insert, update, or delete vectors and metadata directly without relying on an underlying Delta table sync—ideal for real-time applications or custom workflows.


<!-- <img src="../Includes/images/03-vector-search-managed-embeddings.png" alt="Mosaic AI Vector Search managed embeddings method" /> -->

![03-vector-search-managed-embeddings](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-en_us-1.0.1/images/03-vector-search-managed-embeddings.png)

*Figure 5. This diagram shows how Vector Search managed embeddings with automatic sync works.* 

### D4. Governance and Access Control

Mosaic AI Vector Search is governed by **Unity Catalog**, providing a **unified security model for both data and AI assets**. Indexes created in Vector Search appear as securable objects within Unity Catalog, enabling administrators to enforce granular Access Control Lists (ACLs) at the index level. This ensures that only authorized users and applications can query or modify vector data, maintaining consistent security policies across your entire data platform.

## E. Summary

In this lesson, we've explored the complete lifecycle of preparing data for retrieval in a RAG system. We defined **embeddings** as the essential bridge between unstructured text and machine-readable vectors, emphasizing that model selection must align with your specific domain and query patterns. We examined the mechanics of **vector databases**, distinguishing between exact (KNN) and approximate (ANN) search strategies, and discovered how **hybrid search** and **reranking** overcome the limitations of pure vector similarity. Finally, we explored **Mosaic AI Vector Search** and its ability to automate embedding management through **Delta Sync** while providing robust security integration with **Unity Catalog**.

**Key Takeaways:**

1. **Embeddings and Alignment:** Embeddings capture semantic meaning by mapping similar concepts close together in vector space. For effective retrieval, your embedding model must create a shared vector space for both documents and queries—use the same model for both to ensure alignment.
2. **Search Precision:** While **ANN** algorithms deliver the speed and scalability needed for production systems, adding a **reranker** step is often essential to filter noise and ensure high relevance for your language model. Balance the quality improvements against the added latency and cost.
3. **Integrated Architecture:** **Mosaic AI Vector Search** simplifies operations by offering automatic synchronization with **Delta Lake** and supporting flexible ingestion modes (Managed, Self-Managed, or Direct CRUD). This integration eliminates the complexity of managing separate vector database infrastructure while maintaining enterprise-grade governance through Unity Catalog.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>